# 🛠️ Block 6 — Why Won't This Model Train?
## Debugging a real PyTorch training pipeline, one complaint at a time

> **The story.** Last time, your engineering team got a churn-prediction model working in
> PyTorch — a `DataLoader`, a `Module` with its `forward`/`backward`, tensor ops, a training
> loop. It *runs*. The problem is everything **around** it. You've just been made the **manager**
> of that team, and the tickets are piling up:
> 1. *"Training is painfully slow — even though we pay for a GPU."*
> 2. *"Every crash means restarting from scratch — checkpointing keeps breaking."*
> 3. *"The loss won't go down and we can't tell why."*
>
> The engineers keep throwing compute (and money) at it without progress. Your job isn't to
> rewrite their model — it's to **open the hood, find where each problem actually lives, and fix
> it.** That is the real day-to-day of making deep learning work.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- Interactive **quizzes** and **diagrams** build the intuition before the code.
- The team's `DataLoader` and `Module` are **imported, not shown** — we open each one only when a
  complaint leads us to it, just like in real debugging.


## 0. Setup

This notebook is **self-contained**: on Colab the first cell clones the rest of the exercise files
(the `torch_lab.py` training code, the `torch_viz.py` widgets, and the `churn_data.py` loader)
directly from the public course repository. Run the setup cells below in order.

> ✅ The course repo is **public**, so no GitHub token is needed — the cell clones it directly.
> If you're running this notebook from inside a local clone of the repo, the setup cell skips the
> clone and just uses the files already on disk.

> 💻 **GPU runtime.** This notebook is about training speed, so switch Colab to a GPU:
> *Runtime → Change runtime type → T4 GPU*. Everything still runs on CPU (just slower), so it
> won't break if no GPU is available — but you'll get more out of it with one.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"      # public repo — no GitHub token required

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding the `exercises/` helpers — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "exercises", "torch_lab.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/torch_lab.py). On Colab, re-run this cell to clone it; "
        "locally, run the notebook from inside a clone of the FDD-WE0-public repo.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))   # make the helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** Torch is already on Colab; this just adds `einops`
(we'll meet it in Part 4) and pins the rest.

In [ ]:
%pip install -q -r exercises/requirements_block6.txt

**0.3 — Import the libraries** and the team's code. The training utilities live in
**`torch_lab`** (the team's `DataLoader` + `Module`, kept out of sight for now) and the diagrams
& quizzes in **`torch_viz`** so these cells stay about PyTorch, not HTML.

In [ ]:
import importlib
import numpy as np
import torch
import matplotlib.pyplot as plt

import churn_data
import torch_lab
import torch_viz
importlib.reload(churn_data); importlib.reload(torch_lab); importlib.reload(torch_viz)

# 🔒 Reproducibility first. We seed every RNG so the run is repeatable — re-running
# gives the SAME numbers. That is essential when debugging: if results jump around
# between runs, you can't tell whether a change helped or you just got lucky.
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())

> 🔒 **Why seed the RNGs?** We just set `torch.manual_seed` and `np.random.seed` so the
> notebook is **reproducible** — every run produces the same weights, the same shuffling, the
> same loss. When you're hunting a bug, that's non-negotiable: you need to know that a change in
> behaviour came from *your fix*, not from random noise between runs. (Full bit-for-bit
> reproducibility also depends on the **hardware and cuDNN** — there are extra flags like
> `torch.use_deterministic_algorithms(True)` for that — but we won't go down that road here.)

---
# The situation you inherited

The team hands you their setup. We load the churn data through **the same loader the previous
notebook used** (here a stand-in, `churn_data.load_churn`), and build the team's `DataLoader` and
model exactly as they had them. Run it — then we'll start working through the tickets.

In [ ]:
bundle = churn_data.load_churn()
print("customers:", len(bundle["y"]),
      "· numeric features:", bundle["numeric_names"],
      "· categorical:", bundle["categorical_names"])
print("churn rate:", round(float(bundle["y"].mean()), 3))

# The team's DataLoader and model, built just as they had them.
train_loader, val_loader = torch_lab.make_loaders(bundle, batch_size=256)
model = torch_lab.ChurnModel(
    n_numeric=len(bundle["numeric_names"]),
    cat_cardinalities=bundle["cat_cardinalities"])
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print("Model ready — parameters:", sum(p.numel() for p in model.parameters()))

---
# 🐌 Complaint #1 — "Training is so slow, even with a GPU"

Before touching code, get the mental model right. When engineers say training is "slow" they
almost always mean **low throughput**: one **epoch** (a full pass over the training data) takes too
long. Let's build the intuition with a picture — starting with **plain plumbing**, no ML yet.

In [ ]:
torch_viz.water_pipe_analogy()

Hold on to the two facts from that picture: the **narrowest segment sets the speed**, and a
segment can **overflow**. Now ground the exact same pipe in a training step — *that's* all an
epoch is.

In [ ]:
torch_viz.water_pipe_ml()

Now anchor the two ends of the pipe to the hardware. Training data lives in two very
different worlds.

In [ ]:
torch_viz.cpu_gpu_split()

### 🧠 Quick check — did the analogy land?

In [ ]:
torch_viz.mc_quiz("pipe_analogy")

### Where does the model actually live?

The pipe's wide, fast segment is the **GPU**. So the first thing to check: is the team's model
even *on* it? Every PyTorch object reports its device. Let's list what hardware we have, then ask
the model where it's sitting.

In [ ]:
print("Devices PyTorch can see:")
for dev, label in torch_lab.available_devices():
    print(f"  • {dev:8} — {label}")

# Where is the model? (check any parameter's device)
print("\nModel is currently on:", next(model.parameters()).device)

**There it is.** We're paying for a GPU, but the model sits on the **CPU** — the whole pipe
runs through the narrow segment. The fix is the **`.to(device)`** operation: it moves a tensor (or
a whole module) between RAM and VRAM. It isn't free — copying bytes across the CPU↔GPU bus has a
cost — but that cost is tiny next to running the actual math on the right hardware.

### 🎯 Task 1 — move the work to the GPU

Pick the device (the GPU if we have one), then move what needs moving. **Every PyTorch object that
holds numbers can be `.to(device)`'d** — think about *which* objects flow through the training
math each step.

> 📝 **One easy-to-miss detail.** `.to(device)` doesn't move the tensor *where it sits* — it hands
> you back a **new copy** living on the GPU and leaves the original alone. So you must **catch the
> result in a variable** (`x = x.to(device)`); just writing `x.to(device)` on its own does the copy
> and then throws it away. (A handful of PyTorch operations *do* edit in place, but `.to` is not
> one of them — when in doubt, assign the result.)

In [ ]:
device = torch_lab.pick_device()   # 'cuda:0' if a GPU exists, else 'cpu'
print("Training on:", device)

# 🎯 move the MODEL onto the device
model = ???                       # 🎯 hint: the same .to(device) move, on the model

# the training step also needs the DATA on the same device — we'll do that next.
print("Model now on:", next(model.parameters()).device)

The model is on the GPU now — but each **batch** from the DataLoader is still born on the
CPU. Inside the training step, every batch has to be moved too, *before* it meets the model's
weights. Here's the inner loop; fill in the moves.

In [ ]:
def training_step(model, batch, optimizer, device):
    x_num, x_cat, y = batch
    # 🎯 move each piece of the batch onto the device (same .to move as the model)
    x_num = ???
    x_cat = ???
    y     = ???

    logits = model(x_num, x_cat)
    loss = torch.nn.functional.cross_entropy(logits, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

# smoke-test on one batch
loss0 = training_step(model, next(iter(train_loader)), optimizer, device)
print("one step ran, loss =", round(loss0, 4))

### 🧠 Quick check — why move the batches too?

In [ ]:
torch_viz.mc_quiz("why_move_batches")

### The team's "speed-up": fp16 — does it even help?

The engineers tried to go faster by storing weights in **fp16** (16-bit floats) instead of
**fp32** (32-bit). Their reasoning: half the bits → half the bytes → twice as fast. Half the
memory is real. But whether it's *faster* depends entirely on the **hardware**.

In [ ]:
torch_viz.fp32_vs_fp16()

Three things a GPU might do with fp16 numbers:
- **Setting A** — no fp16 units: it pads each fp16 back to fp32 and does the usual fp32 math.
- **Setting B** — no fp16 units, but it packs **two fp16 into one fp32 unit** per cycle.
- **Setting C** — dedicated fp16 units that run **~1.5× faster** than the fp32 ones.

### 🧠 Quiz — in which settings does fp16 actually speed things up?

In [ ]:
torch_viz.mc_quiz("fp16_speedup")

### The flip side of fp16 — vanishing gradients

There's a catch the team didn't see. fp16 spends its 16 bits keeping precision but shrinks the
**exponent** (the *range*). Gradients are often tiny — and below fp16's smallest representable
value they round to **exactly 0**. A zero gradient means **that weight stops learning**. Let's
watch it happen: take a gradient and let each layer of a deep network shrink it (here, simply
**halve it** a few times) — in fp32 it stays alive, in fp16 it falls off a cliff to 0.

In [ ]:
grad_fp32 = torch.tensor(1.0, dtype=torch.float32)
grad_fp16 = torch.tensor(1.0, dtype=torch.float16)

for layer in range(26):          # each "layer" backward halves the gradient
    grad_fp32 = grad_fp32 / 2
    grad_fp16 = grad_fp16 / 2

print(f"after 26 halvings — fp32: {grad_fp32.item():.2e}   ← still a real number, learning continues")
print(f"after 26 halvings — fp16: {grad_fp16.item():.2e}   ← underflowed to 0: the gradient vanished")

Two standard cures:
- **Gradient (loss) scaling** — multiply the loss by a big factor *before* `.backward()`, so the
  gradients are lifted out of the underflow zone; *unscale* them right before the optimizer step.
  PyTorch's `torch.cuda.amp.GradScaler` automates this.
- **bf16** — same 16 bits, but laid out like fp32's *range* (8 exponent bits) at the cost of
  precision. Small gradients simply don't underflow.

### 🧠 Quiz — why do both of these fix the problem?

In [ ]:
torch_viz.mc_quiz("amp_fix")

### 🎯 Task 2 — do mixed precision *properly*

The team's version cast to fp16 with **no** scaler — the worst of both worlds (precision loss,
no safety net). Fix it. The team's `run_epoch` already accepts the right arguments; your job is to
pass them so autocast runs in a **safe dtype** and, if your dtype can underflow, a **scaler**
guards the gradients.

> 💡 You can't be expected to guess the torch spelling, so here are the dtypes to choose between:
> `torch.float32` (full precision), `torch.float16` (fp16) and `torch.bfloat16` (bf16). Now *reason*
> it out from the two cures above: which 16-bit dtype keeps fp32's **range** (and so needs no
> scaler), and which one underflows unless you guard it with a `torch.cuda.amp.GradScaler()`?

In [ ]:
use_amp = device.startswith("cuda")   # autocast pays off on the GPU

# 🎯 pick a SAFE amp dtype, and a scaler ONLY for the dtype that can underflow.
#    dtypes to choose from:  torch.float32 · torch.float16 · torch.bfloat16
amp_dtype = ???        # 🎯 which 16-bit dtype keeps fp32's RANGE? (think back to the two cures)
scaler    = ???        # 🎯 None, or torch.cuda.amp.GradScaler() — only one of the two dtypes needs it

loss, acc = torch_lab.run_epoch(model, train_loader, optimizer=optimizer, device=device,
                                use_amp=use_amp, amp_dtype=amp_dtype, scaler=scaler)
print(f"epoch done — loss {loss:.4f}, acc {acc:.3f}")

### Still too slow — a back-of-the-envelope check

Device fixed, precision fixed… and it's *still* slow. Time to **estimate** how slow it *should*
be and compare. We'll keep it crude on purpose (order-of-magnitude is enough to locate a
bottleneck):
- assume the CPU→GPU transfer is **instant** (we want to expose the *other* bottleneck),
- take a rough GPU rate in **fp16 ops/second**, and convert to an expected epoch time from the
  number of ops the math needs (`batches × (forward + backward + update)`).

If real time ≫ predicted time, the GPU is **starving** — something *upstream* (the dataloader) is
the narrow segment.

In [ ]:
# crude model-side cost: params touched per step, ~3x for fwd+bwd+update
n_params  = sum(p.numel() for p in model.parameters())
n_batches = len(train_loader)
gpu_ops_per_s = 2.0e12          # a deliberately rough fp16 throughput figure
predicted_s = n_batches * (3 * n_params * 256) / gpu_ops_per_s   # 256 = batch size

measured_s = torch_lab.time_epoch(model, train_loader, optimizer, device=device)
print(f"predicted epoch time : ~{predicted_s*1e3:.1f} ms (pure GPU math)")
print(f"measured  epoch time : {measured_s*1e3:.0f} ms")
print(f"\n=> measured is ~{measured_s/max(predicted_s,1e-9):.0f}x slower than the math alone.")
print("   The GPU is starving. Prime suspect: the data path (the DataLoader).")

### Open the hood on the DataLoader

The model and DataLoader were imported, not shown — exactly how you'd inherit them. Now a
complaint points us at the data path, so we open *just that*. `torch_viz.show_source` prints the
code of any imported object, **syntax-highlighted** so it reads like your editor.

In [ ]:
torch_viz.show_source(torch_lab.ChurnDataset.__getitem__, torch_lab._very_costly_operation)

**Found it.** Every `__getitem__` call runs `_very_costly_operation` — a heavy CPU
transform — and it's recomputed for **every sample, every epoch**. That's a needle valve on the
data segment, and it's strangling the GPU.

### 🧠 Quiz — what's the right fix?

In [ ]:
torch_viz.mc_quiz("dataloader_fix")

Right — the transform is identical on the same rows every epoch, so do it **once**, cache
the result, and let the Dataset just read the preprocessed features. The team's Dataset already
supports this with a `heavy=False` flag (think of it as "preprocessing already done and cached").
Rebuild the loaders that way and re-time.

In [ ]:
train_loader, val_loader = torch_lab.make_loaders(bundle, batch_size=256, heavy=False)
fast_s = torch_lab.time_epoch(model, train_loader, optimizer, device=device)
print(f"epoch time, costly op removed from the data path: {fast_s*1e3:.0f} ms"
      f"  ({measured_s/max(fast_s,1e-9):.0f}x faster)")

**Complaint #1 closed.** Model on the GPU, mixed precision done safely, and the dataloader
bottleneck moved out of the hot path. Throughput is no longer the story.

---
# 💾 Complaint #2 — "Every crash means starting over"

Picture a 10-hour run that dies at hour 8 — a GPU fault, a pre-emption, or just a buggy `print`
*after* the loop. Without **checkpointing**, all 8 hours are gone. A checkpoint is a snapshot you
can **resume** from. The team has one, but resuming gives garbage. Let's see why.

### 🧠 First, predict — what carries state we must save?

It's tempting to think "just the weights". But several things evolve *during* training and would
be silently reset on resume. Click every one you think must be saved.

In [ ]:
torch_viz.true_false_quiz("checkpoint_state")

The non-obvious ones are the **optimizer**, the **RNG**, and the **AMP grad scaler** — none
of them look like "the model", yet each quietly carries state across steps:
- **AdamW** doesn't just use the current gradient. It keeps a per-parameter **exponentially moving
  average** of past gradients (and their squares) so it can adapt each weight's step size — that
  running memory *is* the optimizer's state. Drop it and the first post-resume steps lurch while it
  rebuilds from scratch.
- **The AMP grad scaler** (from the fp16 fix) carries a single number: the **current loss-scale
  factor**. It tunes that factor on the fly — nudging it up when steps are safe, backing off when a
  gradient overflows — so it has to remember where it had settled. Reset it and the first steps
  either overflow or underflow until it re-converges.

Here's the team's checkpoint code (first) next to the complete one (second).

In [ ]:
torch_viz.show_source(torch_lab.save_checkpoint_broken, torch_lab.save_checkpoint)

See the difference: the broken version saves **only** `model.state_dict()`. The full one
also stores the optimizer state, the epoch, the AMP scaler, and the RNG state — everything needed
to continue *as if nothing happened*. (How **often** to checkpoint is a trade-off: more frequent =
less lost on a crash, but each save pauses or slows training. The large-scale-training notebook
goes deeper.)

### 🎯 Task 3 — save a proper checkpoint, then resume from it

Train one epoch, snapshot, then *pretend the machine died*: rebuild the model and optimizer from
scratch (random weights, fresh optimizer) and **restore** the checkpoint. If everything that
carries state was saved, the restored loss should match where we left off.

In [ ]:
# train one epoch, then checkpoint
loss_before, _ = torch_lab.run_epoch(model, train_loader, optimizer=optimizer, device=device)
torch_lab.save_checkpoint("ckpt.pt", model, optimizer, epoch=1)
print("checkpoint saved at loss", round(loss_before, 4))

# --- pretend we crashed: fresh model + optimizer ---
model2 = torch_lab.ChurnModel(len(bundle["numeric_names"]), bundle["cat_cardinalities"]).to(device)
optim2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)

# 🎯 restore EVERYTHING from the checkpoint into model2 / optim2.
#    You can't guess torch's spelling, so the call is scaffolded: map_location=device is filled
#    (it sends the loaded tensors to our device). The first three args MIRROR save_checkpoint
#    above — save_checkpoint(path, model, optimizer, ...) — so fill them in that same order.
#    load_checkpoint returns the epoch it was saved at.
resume_epoch = torch_lab.load_checkpoint(???, ???, ???, map_location=device)

# sanity check: evaluate the restored model — should match loss_before, not random
loss_after, _ = torch_lab.run_epoch(model2, val_loader, device=device)
loss_was,   _ = torch_lab.run_epoch(model,  val_loader, device=device)
print(f"resuming at epoch {resume_epoch}: restored val loss {loss_after:.4f} vs original {loss_was:.4f}")
assert abs(loss_after - loss_was) < 1e-4, "State wasn't fully restored!"
print("✅ restored model matches the original — we can resume cleanly")

**Complaint #2 closed.** A crash now costs at most the work since the last checkpoint, and
resuming reproduces the exact training state.

---
# 📉 Complaint #3 — "The loss won't go down and we don't know why"

The deepest one. The model trains, it's fast, it checkpoints — but it doesn't *learn*. And the
team can't say whether it's **over**fitting or **under**fitting, because **they never tracked the
loss.** Step one is always: make the problem visible.

### Two failure shapes to recognize

Before any churn — let's *see* what these words mean on a tiny made-up dataset. We fit curves of
growing flexibility to the same noisy points:

In [ ]:
torch_viz.over_underfit_demo()

- **Underfitting** (left) — the model is too weak (or buggy): it fits *neither* the training
  points nor unseen data. Both train and validation error stay high. Fixes: more capacity, train
  longer, or — find the bug.
- **Good fit** (middle) — it captures the real trend and ignores the noise.
- **Overfitting** (right) — the model memorizes the training points (including their noise):
  **train** error drops toward zero while **validation** error climbs. Fixes: more data,
  regularization, less capacity.

A rough sanity rule of thumb is to have **~10–20× as many training examples as parameters** (take
it with a grain of salt — different architectures have different rules, e.g. Chinchilla's
token-count rule for LLMs). But honestly, **tracking train vs validation loss** tells you far
more than any such rule.

### 🧠 Quiz — read the curves

In [ ]:
torch_viz.true_false_quiz("fit_diagnosis")

### 🎯 Task 4 — make the loss visible

Add the one thing they were missing: record **train** and **validation** loss every epoch, then
plot both. We train for a handful of epochs and watch.

In [ ]:
model = torch_lab.ChurnModel(len(bundle["numeric_names"]), bundle["cat_cardinalities"]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

train_hist, val_hist = [], []
for epoch in range(8):
    tr_loss, _ = torch_lab.run_epoch(model, train_loader, optimizer=optimizer, device=device)
    va_loss, _ = torch_lab.run_epoch(model, val_loader, device=device)   # no optimizer -> eval
    # 🎯 record both losses for this epoch
    ???        # 🎯 hint: append tr_loss to the train_hist list we made above
    ???        # 🎯 hint: append va_loss to the val_hist list

plt.plot(train_hist, label="train", marker="o"); plt.plot(val_hist, label="val", marker="o")
# Honest axis: start at 0 and draw the "random guess" baseline, so a curve that's merely
# *flat at chance* can't masquerade as progress just because matplotlib auto-zoomed in.
plt.axhline(np.log(2), ls="--", color="gray", lw=1, label="random-guess baseline (ln 2 ≈ 0.69)")
plt.ylim(0, max(max(train_hist), max(val_hist), np.log(2)) * 1.15)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.legend(); plt.title("Loss curves"); plt.show()

### Read it — this is the tell

A word on **reading the axis honestly**: matplotlib auto-zooms, so a curve with a tiny wiggle can
*look* like a dramatic descent. That's why we pinned the y-axis to **start at 0** and drew the
**random-guess baseline** (a 2-class model that has learned nothing sits at `ln 2 ≈ 0.69`). With
that reference in view, the verdict is clear: both losses are **stuck right at chance** — the line
is essentially flat at the baseline, not dropping toward 0.

That's the tell. The model has **plenty** of capacity for such a simple dataset (it should easily
overfit), yet it isn't learning at all. Underfitting *with* high capacity and *no* learning is the
signature of a **code bug**, not a modeling choice. Time to open the model — specifically the step
that flattens the categorical embeddings before the MLP.

In [ ]:
torch_viz.show_source(torch_lab.flatten_embeddings)

The categorical features go through embeddings, which then get **reshaped** before the MLP.
Reshapes written as raw `view`/`permute`/`reshape` are notoriously hard to read — you can't tell
from `.view(batch, -1)` *which* axes ended up where. This is exactly where bugs hide. Before we
judge this one, let's get a cleaner lens: **einops**.

### A clearer way to reshape — einops

Every einops call is a little sentence: `pattern = "input axes -> output axes"`. Read it literally:

- **left of `->`** names *every* axis of the input, in order (`b c h w` = batch, channels, height, width);
- **right of `->`** describes the output you want, built from those same names;
- **parentheses `( )` group axes.** On the **output** side, `(c h w)` **merges** those axes into a
  single longer one — stepped through like an **odometer**: the *right-most* name ticks fastest, the
  *left-most* slowest (we'll see this on a picture in a moment). On the **input** side, `(h p1)`
  **splits** one axis into several (you tell it the new sizes, e.g. `p1=2`);
- **changing the order** of the names on the right **permutes** the axes.

Three verbs cover almost everything you'll meet:

| verb | what it does | the raw-PyTorch it replaces |
|---|---|---|
| `rearrange` | move / merge / split axes | `view`, `reshape`, `permute`, `transpose`, `flatten` |
| `reduce` | collapse axes away with an aggregation (`"mean"`, `"sum"`, `"max"`) | `x.mean(dim=…)`, `x.sum(dim=…)`, … |
| `repeat` | add or expand axes by copying | `expand`, `repeat`, `tile` |

We'll build the intuition on **tiny tensors**, each one shown **next to its raw-PyTorch twin** —
so you can see they do the same thing, and that the einops line just *says out loud* what the
cryptic version leaves implicit.

### First, what are these axes? (batch, channel, height, width)

Our examples use little **images**, so before the einops lines, make sure the axis names are
concrete. The one that trips people up is **channel**: a colour image is just **three grids of
numbers** stacked together — a red grid, a green grid, a blue grid.

In [ ]:
torch_viz.channels_intro_viz()

> 🧭 **A trick for reading any multi-dimensional tensor.** When the shape has many axes, mentally
> **peel off the leading axes** until you're looking at a single concrete object you can picture in
> 2-D (one image, one grid). In `(b, c, h, w)`, peel off `b` (pick one image) and `c` (pick one
> colour grid) and you're left with a plain `h × w` table of numbers — easy to reason about.
>
> In particular you'll rarely need to think about the **batch** axis: it just means "many of these,
> processed together", and whatever you do to one sample is done to all of them. So in every example
> below you can safely imagine **`b = 1`** (a single image) and ignore it.

**Example 1 — flatten before an MLP.** You have a little image batch and want one flat feature
vector per image. The raw way is `x.view(x.shape[0], -1)` — but what does `-1` mean, and which
axes get merged? The einops line spells it out.

In [ ]:
from einops import rearrange, reduce

# a 1-image batch, 3 channels, 2x2 pixels, filled 0..11 so we can track every number
img = torch.arange(1 * 3 * 2 * 2).reshape(1, 3, 2, 2)
print("img  (b c h w):", tuple(img.shape))

flat_einops = rearrange(img, "b c h w -> b (c h w)")   # "keep batch, merge c,h,w into features"
flat_manual = img.view(img.shape[0], -1)               # the opaque twin

print("einops 'b c h w -> b (c h w)':", tuple(flat_einops.shape))
print("manual  view(b, -1)          :", tuple(flat_manual.shape))
print("identical result? ", torch.equal(flat_einops, flat_manual))

See *how* the merge happens — which number lands where, and why the colours stay grouped:

In [ ]:
torch_viz.einops_flatten_viz()

**🎯 Your turn (1/3).** Translate a raw reshape into an einops line. Same *shape* of move as
Example 1 — just different axis names.

In [ ]:
# 🎯 flatten a (batch, tokens, dim) tensor into (batch, tokens*dim)
seq = torch.arange(2 * 3 * 4).reshape(2, 3, 4)        # (b=2, tokens=3, dim=4)
your = ???        # 🎯 one rearrange line — keep batch first, merge the other two (see Example 1)
twin = seq.reshape(seq.shape[0], -1)                  # the op you're reproducing
print(tuple(your.shape), "vs", tuple(twin.shape), "— identical?", torch.equal(your, twin))
assert torch.equal(your, twin), "Not yet — keep batch first and merge tokens+dim into one axis."
print("✅ nicely done")

**Example 2 — change the image layout (a permute).** Images from NumPy/PIL usually come as
`(batch, height, width, channels)`, but PyTorch CNNs want `(batch, channels, height, width)`. The
raw way is `x.permute(0, 3, 1, 2)` — correct, but you have to *mentally* track what `0, 3, 1, 2`
means. The einops line names the move.

In [ ]:
hwc = torch.arange(1 * 2 * 2 * 3).reshape(1, 2, 2, 3)   # (b h w c) — NumPy/PIL order
chw_einops = rearrange(hwc, "b h w c -> b c h w")       # say exactly which axes move
chw_manual = hwc.permute(0, 3, 1, 2)                    # same thing, but '0,3,1,2' is a puzzle

print("from (b h w c):", tuple(hwc.shape), "-> (b c h w):", tuple(chw_einops.shape))
print("identical result? ", torch.equal(chw_einops, chw_manual))

Watch the regrouping — each pixel's three channels get scattered into three separate planes:

In [ ]:
torch_viz.einops_permute_viz()

**🎯 Your turn (2/3).** Write a bare `.permute(...)` as a *named* line — the whole point is
that the names say what the numbers can't.

In [ ]:
# 🎯 rewrite x.permute(0, 2, 1) as a named einops rearrange
x = torch.arange(2 * 3 * 4).reshape(2, 3, 4)          # (b=2, t=3, d=4)
your = ???        # 🎯 name the output axes in the order 0,2,1 puts them
twin = x.permute(0, 2, 1)
print(tuple(your.shape), "vs", tuple(twin.shape), "— identical?", torch.equal(your, twin))
assert torch.equal(your, twin), "Not yet — which two axes does 0,2,1 swap? Name them in that order."
print("✅ that reads so much better than '0, 2, 1'")

**Example 3 — aggregate over axes (`reduce`).** Say a CNN gives you `(b, c, h, w)` and you
want **global average pooling**: average away the spatial axes, keep one number per channel. With
`reduce`, the pattern *is* the formula — the axes that disappear from the right are the ones being
averaged. That's the powerful bit: the line reads as the abstraction it computes.

In [ ]:
feat = torch.arange(1 * 3 * 2 * 2, dtype=torch.float32).reshape(1, 3, 2, 2)   # (b c h w)

pooled_einops = reduce(feat, "b c h w -> b c", "mean")   # "average h and w away"
pooled_manual = feat.mean(dim=(2, 3))                     # same, but you must decode dim=(2,3)

print("pooled (b c):", tuple(pooled_einops.shape), "values:", pooled_einops.tolist())
print("identical to feat.mean(dim=(2,3))? ", torch.equal(pooled_einops, pooled_manual))

The axes that vanish from the right are exactly the ones collapsed — see each plane shrink to
one number:

In [ ]:
torch_viz.einops_reduce_viz()

**🎯 Your turn (3/3).** In plain words: *"for each channel, take the **max** over the spatial
positions."* Turn `(b, c, h, w)` into `(b, c)`.

In [ ]:
# 🎯 max-pool the spatial axes away (like Example 3, but not a mean)
feat = torch.arange(1 * 3 * 2 * 2, dtype=torch.float32).reshape(1, 3, 2, 2)
your = ???        # 🎯 same pattern as Example 3; the axes that disappear are the ones reduced
twin = feat.amax(dim=(2, 3))
print("pooled:", tuple(your.shape), "values", your.tolist())
assert torch.equal(your, twin), "Not yet — drop h and w from the right, and pick the right reduction."
print("✅ the pattern IS the formula")

### ⭐ Example 4 (optional challenge — feel free to skip)

This last one is **noticeably harder** than the rest — it's the forward-pass move behind Vision
Transformers, and it chains *several* axis operations at once. If einops already clicked, skip
straight to "Why this matters for our bug" below. If you want to stretch, read on — we'll build it
up **one step at a time**: first *see* the goal, then write the **left** (input) side, then the
**right** (output) side.

**Splitting an image into patches.** This is where raw PyTorch gets genuinely unreadable — *three*
chained ops:

```python
x = x.reshape(b, c, h // p1, p1, w // p2, p2)
x = x.permute(0, 2, 4, 1, 3, 5)
x = x.reshape(b, (h // p1) * (w // p2), c * p1 * p2)
```

einops collapses all three into **one named line** — and you're going to **write that line
yourself**. We won't show it yet; first let's *see* exactly what the move is trying to do.

**First, see the goal.** The blueprint below shows — for a single channel of a single image —
what we're producing: the original 4×6 grid, cut into patches, then rearranged so **each patch
becomes one row** holding all of its pixels. Study it; it's the spec you'll implement. (No einops
line shown — working that out is the challenge.)

In [ ]:
torch_viz.einops_patches_viz(show_pattern=False)

**🎯 Warm-up — the split move.** Writing the **left side** leans on one move: telling einops
that a long axis is really several groups stacked together. Practise it on something easy — a flat
feature vector is really `n` tokens of size `d` glued together; recover the `(b, n, d)` shape
*without* a bare `reshape`. The trick: a **split** like `(n d)` on the **input** side means "this
axis is really `n` groups of `d`", and you pass one of the sizes (`n=3`).

In [ ]:
# 🎯 split the merged feature axis back into (tokens, dim)
flat = torch.arange(2 * 12).reshape(2, 12)            # (b=2, features=12) = 3 tokens x 4 dims
your = ???        # 🎯 put (n d) on the INPUT side and tell einops one of the sizes (n=3)
twin = flat.reshape(2, 3, 4)
print(tuple(your.shape), "vs", tuple(twin.shape), "— identical?", torch.equal(your, twin))
assert torch.equal(your, twin), "Not yet — name the input axis as (n d) and pass n=3."
print("✅ split mastered")

**Step 1 — the left (input) side (your turn).** The image arrives as `b c h w`. To *talk about
patches*, you need to say each spatial axis isn't just pixels but **patches × pixels** — exactly the
split move from the warm-up, only now applied to **both** the height and the width. Try writing just
that input-side split below before reading on (keep `b` and `c` untouched, and don't merge or reorder
anything yet — only expose the patch and pixel axes).

In [ ]:
image = torch.arange(1 * 1 * 4 * 6, dtype=torch.float32).reshape(1, 1, 4, 6)   # (b c h w), 4x6

# 🎯 LEFT SIDE ONLY — the warm-up's split, but applied to BOTH spatial axes:
#    read the height as h patches of p1 pixels, the width as w patches of p2 pixels.
#    Keep b and c untouched; the output just LISTS every axis (nothing merged yet).
split = ???        # 🎯 rearrange(image, "??? -> b c h p1 w p2", p1=2, p2=2)  ← write the input side
print("image", tuple(image.shape), "-> split", tuple(split.shape))   # expect (1, 1, 2, 2, 3, 2)
assert tuple(split.shape) == (1, 1, 2, 2, 3, 2), \
    "Not yet — split the height into (h p1) and the width into (w p2), both with size 2."
print("✅ left side done — both spatial axes are now patches × pixels")

**Step 2 — the right (output) side.** With the input side `b c (h p1) (w p2)` in hand (on the
4×6 image, `p1=2, p2=2` gives `h=2` patch-rows and `w=3` patch-cols → the 6 coloured patches), read
the output straight off the blueprint by answering two questions — ignore the channel for a moment:

- *You want **one row per patch**.* Which two axes **count** the patches? Those go **first**, merged
  into a single axis — that's your number of rows. (Stage ②: how many rows, and from which axes?)
- *Each row must hold all the pixels **inside** its patch.* Which two axes are those pixels? Merge
  them into the second axis — that's the row length.

Finally, **prepend** the channel as `(c p1 p2)` — *not* `(p1 p2 c)`. Order matters here just like in
our bug: a **leading** `c` keeps each channel's pixels **contiguous** (all of red, then all of green,
then all of blue), whereas a **trailing** `c` would **interleave** them pixel-by-pixel (R,G,B, R,G,B,
…) — the same numbers in a scrambled layout. Put the input side and your output side together and
write the one-liner below.

In [ ]:
image = torch.arange(1 * 1 * 4 * 6, dtype=torch.float32).reshape(1, 1, 4, 6)   # (b c h w), 4x6

# 🎯 ONE rearrange line, p1=2, p2=2.
#    Step 1 gave you the INPUT side:  b c (h p1) (w p2)
#    Step 2 is the OUTPUT side — work it out from the blueprint:
#      • one row per patch              -> the two patch-counting axes, merged, come first
#      • each row holds the patch pixels (across channels) -> merge the rest
patches = ???        # 🎯 rearrange(image, "b c (h p1) (w p2) -> ???", p1=2, p2=2)
print("image", tuple(image.shape), "-> patches", tuple(patches.shape))

# the unreadable twin doing the same thing, to check against
manual = (image.reshape(1, 1, 2, 2, 3, 2)   # b c (h p1) (w p2): h=2,p1=2,w=3,p2=2
               .permute(0, 2, 4, 1, 3, 5)   # b h w c p1 p2
               .reshape(1, 2 * 3, 1 * 2 * 2))
assert torch.equal(patches, manual), \
    "Not yet — split both spatial axes, then on the output merge (h w) first, then (c p1 p2)."
print("✅ matched the 3-line reshape+permute+reshape — in ONE named line")

Now that you've written it, here's the same blueprint **with the einops names attached** — so
you can check each group in your line against the picture it produces.

In [ ]:
torch_viz.einops_patches_viz(show_pattern=True)

---
🏁 **End of the optional Example 4.** Everyone rejoins here — whether you did the patch challenge or
skipped it, the next part is the payoff that ties einops back to the team's actual bug.

---
### Why this matters for *our* bug — order is everything

One last tiny example, and it's the one that nails our case. The **same letters in a different
order** on the right give a **different result** — and a raw `.view(...)` would hide that.

In [ ]:
x = torch.arange(2 * 3 * 4).reshape(2, 3, 4)   # (batch=2, tokens=3, dim=4)
print("x:", tuple(x.shape))

# flatten the last two axes, keeping batch first — the intent we usually want:
flat = rearrange(x, "b n d -> b (n d)")
print("b (n d):", tuple(flat.shape))           # (2, 12)  batch stays batch

# the SAME letters, different order = a different result. Naming makes that visible:
swapped = rearrange(x, "b n d -> n (b d)")
print("n (b d):", tuple(swapped.shape))        # (3, 8)   <- batch got mixed in!

Colour each value by the batch it came from and the difference is impossible to miss — the
second pattern interleaves the two batches into every row:

In [ ]:
torch_viz.einops_order_viz()

That last line is the whole point: **`b (n d)` and `n (b d)` have different shapes and
different meaning**, but as a raw `.view(...)` they'd both just be "some reshape" — and if the
numbers happen to line up, a wrong one runs **silently**.

### 🎯 Task 5 — rewrite the flattening with einops and spot the bug

Look again at the team's `flatten_embeddings` (printed above). They stacked the per-column
embeddings on **dim 0** → shape `(n_cat, batch, emb_dim)` → then "fixed a shape error" with
`.view(batch, -1)`. The shape came out right, so nothing crashed — but the **batch axis got mixed
into the features**. Rewrite it the way it was *meant* to be: stack on the batch axis and flatten
with a **named** rearrange.

> 💡 If you tackled the optional Example 4 warm-up, this should ring a bell — it's that same split
> move, just run in the other direction (merging instead of splitting).

In [ ]:
from einops import rearrange

def flatten_embeddings_fixed(embeddings, x_cat):
    embs = [emb(x_cat[:, j]) for j, emb in enumerate(embeddings)]
    # 🎯 stack so the BATCH axis stays first, then flatten (n, d) into one feature axis
    stacked = torch.stack(embs, dim=???)        # 🎯 which dim keeps batch first? -> (batch, n_cat, emb_dim)
    return rearrange(stacked, ???)              # 🎯 merge the (n, d) axes into one, batch first (you've written this exact move before)

# patch it into a fresh model and confirm it learns now
model = torch_lab.ChurnModel(len(bundle["numeric_names"]), bundle["cat_cardinalities"], buggy=False).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

train_hist, val_hist = [], []
for epoch in range(8):
    tr, _ = torch_lab.run_epoch(model, train_loader, optimizer=optimizer, device=device)
    va, _ = torch_lab.run_epoch(model, val_loader, device=device)
    train_hist.append(tr); val_hist.append(va)

plt.plot(train_hist, label="train", marker="o"); plt.plot(val_hist, label="val", marker="o")
plt.axhline(np.log(2), ls="--", color="gray", lw=1, label="random-guess baseline (ln 2 ≈ 0.69)")
plt.ylim(0, max(max(train_hist), max(val_hist), np.log(2)) * 1.15)   # same honest axis as before
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Loss curves — after the fix"); plt.show()
print("final train/val loss:", round(train_hist[-1], 4), "/", round(val_hist[-1], 4))

**There it is — the loss finally drops.** The model was never too weak; one silent reshape
had been scrambling which features belonged to which customer, so the labels never matched the
inputs. `view`/`permute`/`reshape` hid it; **einops made the axis order explicit**, and the bug
became obvious. This is the single highest-leverage habit in this notebook: *name your axes.*

**Complaint #3 closed.** We shipped to production. 🎉

---
# ⭐ Optional — "Inference is eating memory in production"

The model is live, and a new complaint arrives: each request uses **far more memory** than it
should, so the server fits far fewer concurrent requests than expected. This isn't a training
bug — it's an **inference** one.

In [ ]:
torch_viz.autograd_graph()

During the forward pass PyTorch records every op and **keeps the intermediate tensors** so
it can run `.backward()` later. At inference there *is* no backward — so all of that is wasted
VRAM. Let's measure the footprint of a forward pass with grad tracking **on** (the default) versus
**off**.

In [ ]:
def forward_memory_mb(model, batch, device, no_grad):
    x_num, x_cat, y = batch
    x_num, x_cat = x_num.to(device), x_cat.to(device)
    ctx = torch.inference_mode() if no_grad else torch.enable_grad()
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    with ctx:
        out = model(x_num, x_cat)
        # touch the output so the graph (if any) is actually built
        _ = out.sum()
    if device.startswith("cuda"):
        return torch.cuda.max_memory_allocated() / 1e6
    return float("nan")   # CPU: no easy peak-VRAM readout — run this on a GPU runtime to see it

batch = next(iter(val_loader))
with_grad = forward_memory_mb(model, batch, device, no_grad=False)
no_grad   = forward_memory_mb(model, batch, device, no_grad=True)
print(f"peak memory, grad tracking ON : {with_grad:.2f} MB")
print(f"peak memory, inference_mode   : {no_grad:.2f} MB")

> 📏 **Don't be fooled by the tiny gap here.** Our churn model is tiny, so the two numbers come
> out almost identical — there just aren't many intermediates to save. The saving scales with the
> **size and depth of the network and the size of the activations**: on a large model — say an LLM
> with billions of parameters and long sequences — the stored forward graph can **double** the
> memory of a forward pass. There, `inference_mode` is the difference between fitting your model on
> the GPU (and serving many concurrent requests) or going out of memory. Same habit, hugely
> different payoff at scale.

### 🎯 Wrap the production forward pass correctly

The fix is two habits for **every** inference path: put the model in **`model.eval()`** (so layers
like dropout/batchnorm behave deterministically) and run the forward under
**`torch.inference_mode()`** (so no graph is built and the intermediates are freed at once).

In [ ]:
def predict(model, x_num, x_cat, device):
    model.???                       # 🎯 put the model in evaluation mode
    with torch.???:                 # 🎯 the context that disables grad tracking for inference
        logits = model(x_num.to(device), x_cat.to(device))
    return logits.argmax(1)

preds = predict(model, *next(iter(val_loader))[:2], device)
print("predicted", preds.shape[0], "labels with no autograd overhead")

### 🧠 Quiz — the inference-memory story in one question

In [ ]:
torch_viz.mc_quiz("inference_mode")

---
# 🎓 Wrap-up — the manager's debugging checklist

You never rewrote the model. You **located** each problem and fixed it at the source:

| Complaint | Root cause | Fix |
|---|---|---|
| 🐌 Slow training | model & batches on CPU; fp16 without a scaler; costly op in the dataloader | `.to(device)` everything; bf16 (or fp16 + `GradScaler`); precompute & cache the transform |
| 💾 Broken checkpoints | only the weights were saved | also save optimizer / epoch / scaler / RNG, and restore them all |
| 📉 Loss won't drop | a silent `.view()` scrambled the batch axis | name axes with **einops**; the bug became obvious |
| 🚀 Inference OOM | autograd graph kept at inference | `model.eval()` + `torch.inference_mode()` |

**The throughline:** make the problem *visible* (track it, measure it, name your axes), find the
**narrow segment**, and fix *that* — instead of throwing more compute at a pipe that's blocked
somewhere else.